In [4]:
HEADPATH = '/home/gutil/scratch/2025-06-28_Manual-Drift-Recording/'
# HEADPATH = '/home/lag36/scratch/lag36/2024-09-05_lLAG8_EZRDM_Run-01/Drift-Correction/'
# Drift is Barcode data relative to experiment data
# i.e. how much do I have to move the experiment data to align it with the Barcode data

In [2]:
import napari
import numpy as np
from skimage import io

In [ ]:
# Now for stacks
bc_image_first = HEADPATH + 'Barcodes_crop_XY-01_T-Last.tif'
bc_image_last = HEADPATH + 'Barcode_crop_XY-Last_T-Last.tif'

exp_image_first = HEADPATH + 'Experiment_crop_XY-01_T-Last.tif'
exp_image_last = HEADPATH + 'Experiment_crop_XY-Last_T-Last.tif'

# Load the images
bc_image_first_data = io.imread(bc_image_first)
bc_image_last_data = io.imread(bc_image_last)
exp_image_first_data = io.imread(exp_image_first)
exp_image_last_data = io.imread(exp_image_last)

# Cropping parameters (Y, X)
crop_start = (600, 1000)  # Top-left corner
crop_end = (600+550, 1000+400)  # Bottom-right corner

# Crop the images
bc_image_first_cropped = bc_image_first_data[crop_start[0]:crop_end[0], crop_start[1]:crop_end[1]]
bc_image_last_cropped = bc_image_last_data[crop_start[0]:crop_end[0], crop_start[1]:crop_end[1]]
exp_image_first_cropped = exp_image_first_data[crop_start[0]:crop_end[0], crop_start[1]:crop_end[1]]
exp_image_last_cropped = exp_image_last_data[crop_start[0]:crop_end[0], crop_start[1]:crop_end[1]]

bc_stack = np.stack([bc_image_first_cropped, bc_image_last_cropped])
exp_stack = np.stack([exp_image_first_cropped, exp_image_last_cropped])

# Open Napari viewer
viewer = napari.Viewer()

# Add the first image stack
layer_barcodes = viewer.add_image(np.stack(bc_stack),
                          name='Barcodes',
                          opacity = 1,
                          contrast_limits=(0, 5000))

# Add the second image stack
layer_experiment = viewer.add_image(np.stack(exp_stack),
                          name='Experiment',
                          opacity = 0.5,
                          contrast_limits=(300, 2000),
                          colormap='red')

drift_metadata = {}  # Dictionary to store drift metadata

# Function to record drift after manual alignment
def record_drift(event):
    
    current_fov = viewer.dims.current_step[0]  # Get the current FOV index
    translation = layer_experiment.affine.translate
    drift_metadata[current_fov] = translation#(translation[0], translation[1])  # Store as (dy, dx)
    

last_fov = viewer.dims.current_step[0]  # Store initial FOV

def on_fov_change(event):
    global last_fov
    current_fov = viewer.dims.current_step[0]
    
    if current_fov != last_fov:
        # FOV has changed!
        if current_fov not in drift_metadata:
            layer_experiment.affine.translate = [0., 0., 0.]
        else:
            layer_experiment.affine.translate = drift_metadata[current_fov]

        layer_experiment.refresh()
        last_fov = current_fov  # Update last_fov to current


layer_experiment.events.affine.connect(record_drift)
# layer_experiment.metadata['drift_correction'] = drift_metadata
# viewer.dims.events.point.connect(reset_drift)
viewer.dims.events.current_step.connect(on_fov_change)

napari.run()

In [90]:
print(drift_metadata)

{0: array([   0.        ,  -86.84210635, -121.0526331 ]), 1: array([  0.        , 169.73684424,  67.105264  ])}


### Nd2 reader for experiment data


In [4]:
HEADPATH = '/home/lag36/scratch/lag36/2024-09-05_lLAG8_EZRDM_Run-01/Drift-Correction/'

In [25]:
nd2_filename = HEADPATH + 'Experiment_crop_XY-All_T-Last.nd2'

In [26]:
from nd2reader import ND2Reader

In [ ]:
with ND2Reader(nd2_filename) as nd2file:
    print(nd2file.get_frame_2D(t=0, v=626))

In [ ]:
positions

[[401 409 379 ... 369 366 415]
 [358 355 375 ... 412 370 406]
 [391 415 373 ... 352 395 394]
 ...
 [424 382 397 ... 375 406 414]
 [376 423 405 ... 392 372 401]
 [439 417 381 ... 365 382 401]]
